In [1]:
import torch
import torch.nn.functional as F
from datasets import load_dataset, load_dataset_builder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt # for making figures
%matplotlib inline
import tiktoken
import pyarrow as pa
import pyarrow.compute as pc
import os

/home/taihim/projects/YA-GPT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("../data/pg100.txt", "r") as f:
  shakespeare_text = (f.read())

FileNotFoundError: [Errno 2] No such file or directory: '../data/pg100.txt'

In [3]:
encoding = tiktoken.get_encoding("cl100k_base")

In [4]:
encoded_text = encoding.encode(shakespeare_text)

NameError: name 'shakespeare_text' is not defined

In [5]:
n = int(0.9*len(encoded_text))
train_data = torch.tensor(encoded_text[:n])
val_data = torch.tensor(encoded_text[n:])

print(f"Training data shape: {train_data.shape}")
print(f"Val data shape: {val_data.shape}")

Training data shape: torch.Size([1312907])
Val data shape: torch.Size([145879])


In [6]:
context_size = 16
batch_size = 4

In [7]:
torch.manual_seed(1337)
def get_batch(split="train"):
  data = train_data if split=="train" else val_data
  idx = torch.randint(len(data) - context_size, (batch_size,), )
  x = torch.stack([data[i:i+context_size] for i in idx])
  y = torch.stack([data[i+1:i+context_size+1] for i in idx])
  return x, y

xb, yb = get_batch()
print(xb.shape)
print(yb.shape)
print(xb[0])
print(yb[0])

print("-----" * 15)
print("")

for i in range(batch_size):
  for l in range (context_size):
    print(f"When input is {xb[i, :l+1].tolist()}, target is {yb[i, l]}")

torch.Size([4, 16])
torch.Size([4, 16])
tensor([26236, 68603,    26,   369,   433,  5084,   198,  7009, 81413,   311,
          387, 30526,    13, 10699,   374,   420])
tensor([68603,    26,   369,   433,  5084,   198,  7009, 81413,   311,   387,
        30526,    13, 10699,   374,   420,   198])
---------------------------------------------------------------------------

When input is [26236], target is 68603
When input is [26236, 68603], target is 26
When input is [26236, 68603, 26], target is 369
When input is [26236, 68603, 26, 369], target is 433
When input is [26236, 68603, 26, 369, 433], target is 5084
When input is [26236, 68603, 26, 369, 433, 5084], target is 198
When input is [26236, 68603, 26, 369, 433, 5084, 198], target is 7009
When input is [26236, 68603, 26, 369, 433, 5084, 198, 7009], target is 81413
When input is [26236, 68603, 26, 369, 433, 5084, 198, 7009, 81413], target is 311
When input is [26236, 68603, 26, 369, 433, 5084, 198, 7009, 81413, 311], target is 387
Whe

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Combine the `Head` and `MultiHeadAttention` into one class that processes all the heads in parallel,
# treating the heads as another batch dimension


# 64, 256, 512
# 2 head attention -> 64, 256, 256
# after concat -> 64, 256, 512

# how to do this in parallel?
# treat each head as a batch dimension
# if 2 heads
# 64, 2, 256, 256?
# then combine?

# q,k,v wont be small size then i.e. they will be embd_size, embd_size instead of embd_size, embd_size // num_heads
# first we reshape from B, T, C to B, T, num_heads, embd_size // num_heads (i.e. head_size)
# then we transpose num_heads and T dimension to get B, num_heads, T, head_size
# we carry out all operations on this matrix
# at the end we reshape back to B, T, C

import math
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

embd_size = 384
max_len = 256
block_size = 256
num_blocks = 6
dropout = 0.1

class MultiHeadAttention(nn.Module):
  def __init__(self, n_embd, n_heads):
    super().__init__()
    self.n_embd = n_embd
    self.n_heads = n_heads
    self.head_size = n_embd // n_heads

    self.Wk = nn.Linear(n_embd, n_embd)
    self.Wq = nn.Linear(n_embd, n_embd)
    self.Wv = nn.Linear(n_embd, n_embd)

    self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))
    self.proj = nn.Linear(n_embd, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape

    k = self.Wk(x)
    q = self.Wq(x)
    v = self.Wv(x)

    k = k.view(B, T, self.n_heads, self.head_size)
    q = q.view(B, T, self.n_heads, self.head_size)
    v = v.view(B, T, self.n_heads, self.head_size)

    k = k.transpose(2, 1)
    q = q.transpose(2, 1)
    v = v.transpose(2, 1)


    wei = q @ k.transpose(-2, -1) / math.sqrt(self.head_size)
    wei = wei.masked_fill(self.tril[:, :, :T, :T] == 0, float("-inf"))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)

    out = wei @ v

    out = out.transpose(1, 2).reshape(B, T, C)

    out = self.dropout(self.proj(out))

    return out

class Ffwd(nn.Module):
  def __init__(self, embd_size, head_size, n_heads):
    super().__init__()
    # self.ffwd = nn.Sequential(nn.Linear(embd_size, head_size * n_heads), nn.ReLU(), nn.Linear(head_size * n_heads, embd_size), nn.Dropout(dropout))
    # gemini suggested expanding linear layer 512 * 4 = 2048
    # self.ffwd = nn.Sequential(nn.Linear(embd_size, 4 * embd_size), nn.ReLU(), nn.Linear(4 * embd_size, embd_size), nn.Dropout(dropout))
    self.ffwd = nn.Sequential(nn.Linear(embd_size, 4 * embd_size), nn.GELU(), nn.Linear(4 * embd_size, embd_size), nn.Dropout(dropout))

  def forward(self, x):
    return self.ffwd(x)


class Block(nn.Module):
  def __init__(self, embd_size, n_heads):
    super().__init__()
    self.ln1 = nn.LayerNorm(embd_size)
    self.mh_attn = MultiHeadAttention(embd_size, n_heads)
    self.ln2 = nn.LayerNorm(embd_size)
    self.ffwd = Ffwd(embd_size, embd_size // n_heads, n_heads)

  def forward(self, x):
    out = x + self.mh_attn(self.ln1(x))
    out = out + self.ffwd(self.ln2(out))

    return out


class AttentionLanguageModel(nn.Module):
  def __init__(self, vocab_size, n_heads, num_blocks):
    super().__init__()
    self.embedding_table = nn.Embedding(vocab_size, embd_size)
    self.position_embedding_table = nn.Embedding(max_len, embd_size)

    self.blocks = nn.ModuleList([Block(embd_size, n_heads) for _ in range(num_blocks)])

    self.ff = nn.Linear(embd_size, vocab_size)


  def forward(self, idx, target=None):
    _, T = idx.shape
    logits = self.embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))

    out = logits
    for block in self.blocks:
      out = block(out)

    out = self.ff(out)

    loss = None
    if not target is None:
      B,T,C = out.shape
      out = out.view(B*T, C)

      targets = target.view(B*T)
      loss = F.cross_entropy(out, targets)

    return out, loss

  # def generate(self, ctx, max_tokens):
  #   for _ in range(max_tokens):
  #     logits, _ = self(ctx)

  #     # get only last timestep becuase thats the prediction for whats next
  #     # we already know the preceding chars

  #     logits = logits[:, -1, :]

  #     probs = F.softmax(logits, dim=1)
  #     generated_token = torch.multinomial(probs, num_samples=1)
  #     ctx = torch.cat((ctx, generated_token), dim=1)

  def generate(self, ctx):
      # ctx will be a 1, 1 tensor
      # we will pass this to the model and get a 1, 1, 384 tensor that will then go on to give 1, 1, 50257 logits
      # we will then do softmax on the logits and pick the highest probability. or use torch.multinomial
      logits, _ = self.forward(ctx)
      probs = torch.softmax(logits[:, -1, :], dim=-1)
      topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)

      output = torch.multinomial(topk_probs, num_samples=1)

      xcol = torch.gather(topk_indices, -1, output)
      # output = output

      return torch.cat((ctx, xcol), dim=1)

# m = AttentionLanguageModel(len(pretrain_vocab), 8, num_blocks).to(device)
m = AttentionLanguageModel(encoding.n_vocab, 8, num_blocks).to(device)

xb = xb.to(device)
output, _ = m(xb)
print(output.shape)

torch.Size([16, 16, 100277])


In [5]:
state_dict = torch.load('my_shakespeare_model3.pth', weights_only=True)
m.load_state_dict(state_dict)

FileNotFoundError: [Errno 2] No such file or directory: 'my_shakespeare_model3.pth'

In [40]:
for p in m.parameters():
  print(p.shape)

optimizer = torch.optim.AdamW(m.parameters(), lr=3e-4)
eval_iters = 200
def estimate_loss(model):
    device = next(model.parameters()).device
    model.eval()
    losses = {'train': 0.0, 'val': 0.0}
    for split in ['train', 'val']:
        total_loss = 0.0
        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            xb = xb.to(device)
            yb = yb.to(device)
            _, loss = model(xb, yb)
            total_loss += loss.item()
        losses[split] = total_loss / eval_iters
    model.train()
    return losses

torch.Size([100277, 384])
torch.Size([256, 384])
torch.Size([384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384])
torch.Size([384])
torch.Size([1536, 384])
torch.Size([1536])
torch.Size([384, 1536])
torch.Size([384])
torch.Size([384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384])
torch.Size([384])
torch.Size([1536, 384])
torch.Size([1536])
torch.Size([384, 1536])
torch.Size([384])
torch.Size([384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384, 384])
torch.Size([384])
torch.Size([384])
torch.Size([384])
torch.Size([1536, 384])
torch.Size([1536])
torch.Size([384, 1536])
torch.

In [10]:
# parallel multi head attn (4 heads, 128 head size, 512 embedding)
# more ram usage but faster training
# 8 epochs in 30mins instead of 7
# 12gb ram instead of 7.5gb
batch_size = 16
epochs = 50000
eval_interval = 500

for i in range(epochs):
  xb, yb = get_batch('train')

  xb = xb.to(device)
  yb = yb.to(device)

  # if i % eval_interval == 0:
  #   losses = estimate_loss(m)
  #   print(f"Train loss {losses['train']} and eval loss: {losses['val']}")

  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  
  if i % eval_interval == 0:
    print(f"Train loss {loss}")

    
  loss.backward()

  optimizer.step()

print(loss)

NameError: name 'optimizer' is not defined

In [54]:
torch.manual_seed(42)
inp = torch.tensor(encoding.encode("Duck"), dtype=torch.int).view(1, -1).to(device)
print(inp.size())
max_len = 200
while inp.size(1) < max_len:
    inp = m.generate(inp)

print(encoding.decode(inp.tolist()[-1]))

torch.Size([1, 2])
Duckes.
Scene IV. On board Pericles’ ship.
Bmish savage horse that victory most come to thee, thou dost thou that hither left away! Speak and add once more hereafter to thy brother is mine own shame to my slave; would he dothriftly order of
To tell the boats of peace tomorrow your hands and make thee to thine. How strange a lie;
The Queen Hecomes a grievouslyright, thy mother living liv’Twice, beingestable; the dull blood ere me, go to thee.
O Antony goes not a fever burns thee of peace!

 Enter Caesar cover’d from him.
Kingous moodyery at thy brother of England’s better comfort.

 [_Exit Puc’d presence still some half that that Thane._]

 Enter Diom it, dear Cassiolets; tell him; thou should from thee?

Lies, for my consent
Is straight his body.
What his aff meat these ill reporter


In [17]:
# pretraining gpt generation with tiktoken encodings 1 runs 4.58 train loss 5.95 val loss

# gen_start = torch.zeros((1, 1), dtype=torch.long).to(device)
gen_start = torch.tensor([[51]], dtype=torch.long, device=device)
output = "".join([encoding.decode([idx]) for idx in m.generate(gen_start, max_tokens=256).squeeze().tolist()])
print(output)

TAtt education his midnight, ’tis for France;
And thunders break him in his bend his bed-favish brass, what temper bids whereof jesting his height
iers as man, and if the Trebona charm tomorrow guard shall be silent, and his fixed outlaws George’s retire his quillets patience after Humphrey be said Henry will not to thy body’s staff I will lay; and all his outwardured king, not a schoolmaster
Of mind is delivered.
IUSleth teachars dæ

 ACT these crystal be straying father and vassail, clapped organs, butchers to their thyenny. Come to adventurous fortcount piece of morefeitings did discords, walk so many vows scores of thither will prove loyal no silk, and more quarrel.

Might be
As England would confinour,
The ritesUDEMETace about none of thy tables cheeks;
His love would be dismissed it was’t beforeen but what it except motive embrace before thee! Othron star to disgrace new comfort.

TIT fleetingTITANNESTOLIVERse look for evernew Olivia?

S feet stumbled. And hath forward winds blow

In [11]:
import tiktoken

# Replace 'gpt2' with whichever encoding you loaded for your dataset
encoding = tiktoken.get_encoding("cl100k_base") 

# Encode a simple newline character
newline_token = encoding.encode("T")

print(newline_token) 
# Output: [198]

[51]


In [15]:
torch.save(m.state_dict(), 'my_shakespeare_model3.pth')

In [17]:
m.state_dict()

OrderedDict([('embedding_table.weight',
              tensor([[ 0.2102,  0.1764, -0.1929,  ...,  0.3240, -0.7828,  0.4224],
                      [ 0.2869,  0.9474,  0.3475,  ...,  0.7006, -0.9887,  1.1510],
                      [-0.5258,  0.0562, -0.6987,  ...,  0.4756, -0.5117,  0.6649],
                      ...,
                      [-0.1153,  0.4327,  0.5770,  ...,  1.0439,  0.1835,  1.0049],
                      [-0.2814, -1.0712, -0.1780,  ...,  0.4460,  0.1248, -0.2517],
                      [ 0.2132, -0.8333,  0.9040,  ..., -1.2761, -0.0974, -0.6760]],
                     device='cuda:0')),
             ('position_embedding_table.weight',
              tensor([[ 0.6957,  0.2143,  1.8782,  ...,  0.0277,  0.2418,  0.0633],
                      [ 0.3319, -1.0005,  0.3136,  ..., -0.7054,  0.7541, -0.3092],
                      [ 0.6938,  0.0074,  0.3758,  ..., -0.6499,  0.4954, -1.0199],
                      ...,
                      [ 0.2608,  1.7772,  0.8620,  ..., -0.1

In [2]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

@dataclass
class GPTConfig:
    block_size: int = 256
    vocab_size: int = 65
    n_layer: int = 6
    n_heads: int = 6
    n_embd: int = 384

config = GPTConfig()

kqv = nn.Linear(config.n_embd, 3 * config.n_embd)

mask = torch.tril(torch.ones(config.block_size, config.block_size))
mask.view(1, 1, config.block_size, config.block_size)

tensor([[[[1., 0., 0.,  ..., 0., 0., 0.],
          [1., 1., 0.,  ..., 0., 0., 0.],
          [1., 1., 1.,  ..., 0., 0., 0.],
          ...,
          [1., 1., 1.,  ..., 1., 0., 0.],
          [1., 1., 1.,  ..., 1., 1., 0.],
          [1., 1., 1.,  ..., 1., 1., 1.]]]])

In [49]:
kek = kqv(torch.ones((1, 4, config.n_embd)))
kek.split(384, dim=2)[2].shape

torch.Size([1, 4, 384])

In [30]:
kek[0].shape

torch.Size([1152])

In [51]:
mask.shape

torch.Size([256, 256])

In [52]:
mask

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 1., 0., 0.],
        [1., 1., 1.,  ..., 1., 1., 0.],
        [1., 1., 1.,  ..., 1., 1., 1.]])

In [62]:
mask[:255, :255]

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 1., 0., 0.],
        [1., 1., 1.,  ..., 1., 1., 0.],
        [1., 1., 1.,  ..., 1., 1., 1.]])

In [64]:
mask.view(1, 1, config.block_size, config.block_size).shape

torch.Size([1, 1, 256, 256])